# MCP using local servers

In this notebook, we will see
- Chain of Thought prompting for tools
- Running a local http mcp server
  - Connecting to http server as a client
  

Inside the `mcp_servers` folder, we have another mcp tools file `mcp_http_server.py`.

In this python file or mcp server file, we have two mcp tools
1. the `read_local_file` mcp tool - which is same as the one used in notebook [1_intro_to_MCP](1_intro_to_MCP.ipynb)
2. A new mcp tool called `execute_python_code`
   1. We can use this tool to directly run python code

We will combine both tools together.
1. We will read a python script from local machine via the mcp tool `read_local_file`
2. Then we will send the output of the first tool to the second tool which is `execute_python_code`
3. We will us Chain of Thought prompting for this execution

In [1]:
import ollama
from mcp import ClientSession
from mcp.client.sse import sse_client as http_client

In [2]:
LOCAL_SERVER_URL = "http://127.0.0.1:8000/sse"

In [3]:
SYSTEM_PROMPT = (
                "You are a helpful assistant with access to tools. "
                "If a task requires multiple steps (e.g., reading a file then running it), "
                "perform them one by one. Use the output of one tool to inform the next. "
                "If you need to read a file, use the file reading tool first. If you need to execute code, use the code execution tool."
                "After getting result from executing code, give the output as the final answer."
                "When you have the final answer, summarize it for the user."
                "If the python files have syntax error then output the syntax error as the final answer without trying to execute it and tell what the error is"
            )

In [6]:
async def run_chainable_agent(user_prompt, system_prompt, model):
    # Define the conversation history
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt}
    ]

    async with http_client(LOCAL_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 1. Fetch and format tools for Ollama
            tools_result = await session.list_tools()
            ollama_tools = [{
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.inputSchema,
                }
            } for t in tools_result.tools]

            print(f"🚀 Starting agent loop for: '{user_prompt}'")
            
            # 2. Start the Loop (Max 5 turns to prevent infinite loops)
            for turn in range(5):
                print(f"\n--- Turn {turn + 1} ---")
                
                response = ollama.chat(
                    model=model,
                    messages=messages,
                    tools=ollama_tools
                )
                
                assistant_msg = response['message']
                messages.append(assistant_msg) # Record the model's reasoning/call

                # Check if the model wants to call tools
                if not assistant_msg.get('tool_calls'):
                    print("✅ Final answer reached.")
                    return assistant_msg['content']

                # 3. Execute Tool Calls
                for tool_call in assistant_msg['tool_calls']:
                    t_name = tool_call['function']['name']
                    t_args = tool_call['function']['arguments']
                    
                    print(f"🛠️ Tool Call: {t_name}({t_args})")
                    
                    # Execute the tool via MCP
                    result = await session.call_tool(t_name, t_args)
                    
                    # Append the result back to the conversation
                    messages.append({
                        'role': 'tool',
                        'content': str(result.content),
                        'name': t_name
                    })
                    print(f"📥 Tool Result received.")

            return "⚠️ Loop limit reached without a final answer."



In [8]:
query = "Read fibonacci_numbers.py in the scripts folder and then execute it. Path to file=scripts/fibonacci_numbers.py"
MODEL = "llama3.2:3b"

result = await run_chainable_agent(query, SYSTEM_PROMPT, MODEL)
print(f"\nRESULT:\n{result}")

🚀 Starting agent loop for: 'Read fibonacci_numbers.py in the scripts folder and then execute it. Path to file=scripts/fibonacci_numbers.py'

--- Turn 1 ---
🛠️ Tool Call: execute_python_code({'file_path': 'scripts/fibonacci_numbers.py'})
📥 Tool Result received.

--- Turn 2 ---
✅ Final answer reached.

RESULT:
The Fibonacci numbers calculated by the Python script are: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]. 

Would you like to execute another file or ask a different question?


## Error message with code cell above: 

This is a completely different type of error than the ones we fixed earlier.

Look at the very last line of the traceback:
`httpx.ConnectError: All connection attempts failed`

### What is the problem?

You are no longer using the `stdio_client` (which runs a local script). You are now using the `http_client` (Server-Sent Events / SSE), which tries to connect over the network to a server at `http://127.0.0.1:8000/sse`.

The error means **the Jupyter cell is trying to call a server that is not currently running.**

### The Fix

As the markdown text in your notebook explains right below that cell, you must manually start the HTTP server in a separate terminal before running the Jupyter cell.

Here is exactly what you need to do:

1. **Open a new Terminal window** (keep your Jupyter Notebook open).
2. Make sure your Python virtual environment is activated (`.venv`).
3. Run this exact command to start the server:
```bash
python mcp_servers/mcp_http_server.py

```


4. You should see a message in that terminal saying the server is running on port 8000. **Do not close this terminal.** Leave it running in the background.
5. Go back to your Jupyter Notebook and run the cell again:
```python
result = await run_chainable_agent(query, SYSTEM_PROMPT, MODEL)
print(f"\nRESULT:\n{result}")

```



Because the server is now awake and listening on `127.0.0.1:8000`, the connection will succeed, and your agent will start chaining tools!

When we run the cell above we get an error because our mcp tool is not active on the server. For this we first have to start the local server. 
To do this:
1. Open terminal
2. Navigate to this repository
3. Activate your virtual environment
4. Run `python mcp_servers/mcp_http_server.py`


After executing this command, you will see that the server has started and now actively listening for connections

#### Check if the server is listening and we can connect to the server
1. Open another terminal and navigate to the repository
2. Activate the virtual environment
3. Run `curl -v http://127.0.0.1:8000/sse`
4. If the connection is established to the server, you should see something like `HTTP/1.1 200 OK` on your terminal
5. On the server side, that is the terminal window where server is running, you will also see `200 OK` if successfully connected
6. Exit the curl command on the client side with `control + c/d`


#### Connect LLM to the MCP server
Re-ru the above cell again, to see if we can connect to the server


## Further explanation: 

That is a beautiful sight! You officially have a working HTTP-based Model Context Protocol (MCP) server running locally, and your agent successfully connected to it over the network.

As an AI Product Manager, I want to draw your attention to something incredibly fascinating in those logs. Did you notice how the agent behaved?

Your prompt was:

> *"**Read** fibonacci_numbers.py in the scripts folder and **then execute** it."*

But look at the agent's actual steps:

```text
--- Turn 1 ---
🛠️ Tool Call: execute_python_code({'file_path': 'scripts/fibonacci_numbers.py'})

```

### The PM Insight: Agentic "Laziness" (or Efficiency!)

The LLM completely skipped the "read" tool and went straight to the "execute" tool!

Why? Because autonomous agents are highly goal-oriented. The model realized that the `execute_python_code` tool accepts a file path directly. It reasoned: *"Why waste a turn reading the text of the file when my ultimate goal is just to execute it and get the output?"*

This is a classic example of why testing different models is so critical for AI products. A 3B model might take a shortcut like this, whereas a massive 70B model might strictly follow the chain-of-thought instructions to the letter (Read $\rightarrow$ Execute $\rightarrow$ Summarize). Both get the job done, but understanding these behavioral quirks is a huge part of your job when designing agentic workflows.

You have reached the **DIY - Do It Yourself** section at the bottom of your notebook. What kind of custom tool are you thinking about building and adding to your `mcp_http_server.py` file to test it out?

## DIY - Do It Yourself

Now try to implement new mcp tools and make them work for you. You can start with very simple math tools.


# Example 1

The Business Problem: Utility companies get thousands of calls asking, "Why is my electricity bill so high this month?" Customer support agents have to manually dig through databases.
The AI PO Solution: An AI agent that uses an MCP tool to fetch a customer's Smart Meter data, analyzes the consumption peaks, and explains it in simple terms.

Why it impresses Enercity: It demonstrates cost-reduction in customer operations and showcases how AI can handle secure, personalized user data without the LLM having raw access to the entire database.

The MCP Tool Implementation (Add this to your server file):

# Add this tool to your mcp_http_server.py
@mcp.tool()
def fetch_smart_meter_data(customer_id: str, month: str) -> str:
    """
    Fetches the daily electricity consumption (in kWh) for a specific customer.
    Use this to help customers understand their energy bill.
    """
    # In a real Enercity system, this would query an SQL database or SAP system.
    # For your portfolio, we use mock data.
    mock_database = {
        "CUST-8472": {
            "January": {"total_kwh": 450, "peak_usage_appliance": "Heat Pump", "peak_time": "18:00 - 21:00"},
            "February": {"total_kwh": 380, "peak_usage_appliance": "Electric Vehicle Charger", "peak_time": "23:00 - 05:00"}
        }
    }
    
    data = mock_database.get(customer_id, {}).get(month)
    if not data:
        return f"Error: No smart meter data found for customer {customer_id} in {month}."
    
    return (
        f"Data for {customer_id} in {month}:\n"
        f"- Total Consumption: {data['total_kwh']} kWh\n"
        f"- Main Driver: {data['peak_usage_appliance']}\n"
        f"- Peak Hours: {data['peak_time']}"
    )

## Run the next code cell for Example 1


In [11]:
query = "Customer CUST-8472 is complaining about their January bill. Look up their smart meter data and write a friendly email explaining what drove their usage up."
MODEL = "llama3.2:3b"

result = await run_chainable_agent(query, SYSTEM_PROMPT, MODEL)
print(f"\nRESULT:\n{result}")

🚀 Starting agent loop for: 'Customer CUST-8472 is complaining about their January bill. Look up their smart meter data and write a friendly email explaining what drove their usage up.'

--- Turn 1 ---
🛠️ Tool Call: fetch_smart_meter_data({'customer_id': 'CUST-8472', 'month': 'January'})
📥 Tool Result received.

--- Turn 2 ---
✅ Final answer reached.

RESULT:
Subject: Explanation of Your January Bill

Dear CUST-8472,

I hope this email finds you well. I am writing to address your concern about the recent bill you received for January. After reviewing your smart meter data, I'd like to explain what might have contributed to the increase in your usage.

According to your data, your total consumption in January was 450 kWh, which is slightly higher than usual. The main driver behind this increase was the heat pump, which accounted for a significant portion of your energy usage. This makes sense, as January is typically one of the colder months, and heat pumps are often used more during thi

# Example 2

The "Solar Yield Predictor"

The Business Problem: Enercity sells photovoltaic (solar) solutions to homeowners. Customers constantly want to know how much power they will generate tomorrow so they can plan when to charge their EV or run the washing machine.
The AI PO Solution: An MCP tool that calculates solar energy yield based on a location's weather forecast and the customer's solar panel capacity.

Why it impresses Enercity: It aligns perfectly with their green energy products, promotes grid stability (shifting consumption to high-yield times), and shows you understand API chaining (combining weather APIs with mathematical logic).

The MCP Tool Implementation:

# Add this tool to your mcp_http_server.py
@mcp.tool()
def predict_solar_yield(city: str, panel_capacity_kwp: float) -> str:
    """
    Calculates the expected solar energy generation (in kWh) for the next day.
    Requires the city name and the size of the customer's solar installation in kWp.
    """
    # Mocking a weather API call (e.g., OpenWeatherMap)
    weather_conditions = {
        "Hannover": {"sun_hours": 3.5, "condition": "Cloudy"},
        "Munich": {"sun_hours": 8.0, "condition": "Sunny"}
    }
    
    forecast = weather_conditions.get(city, {"sun_hours": 5.0, "condition": "Partly Cloudy"})
    
    # Simple physics estimation: Capacity * Sun Hours * 0.75 (Efficiency Loss)
    estimated_kwh = panel_capacity_kwp * forecast["sun_hours"] * 0.75
    
    return (
        f"Tomorrow's forecast for {city}: {forecast['condition']} ({forecast['sun_hours']} sun hours).\n"
        f"A {panel_capacity_kwp} kWp system is expected to generate approximately {estimated_kwh:.2f} kWh tomorrow."
    )

In [13]:
query = "I live in Hannover and have a 10 kWp solar system on my roof. Calculate my expected solar yield for tomorrow and tell me if it's enough to fully charge my 50 kWh electric car battery."
MODEL = "llama3.2:3b"

result = await run_chainable_agent(query, SYSTEM_PROMPT, MODEL)
print(f"\nRESULT:\n{result}")

🚀 Starting agent loop for: 'I live in Hannover and have a 10 kWp solar system on my roof. Calculate my expected solar yield for tomorrow and tell me if it's enough to fully charge my 50 kWh electric car battery.'

--- Turn 1 ---
🛠️ Tool Call: predict_solar_yield({'panel_capacity_kwp': '10', 'city': 'Hannover'})
📥 Tool Result received.

--- Turn 2 ---
✅ Final answer reached.

RESULT:
Based on the tool output, it appears that your 10 kWp solar system can expect to generate around 26.25 kWh of electricity tomorrow.

To determine if this is enough to fully charge your 50 kWh electric car battery, let's do some simple calculation:

Assuming a standard charging efficiency of 95% (which means 5% of the energy is lost during charging), we can calculate the maximum amount of energy that can be charged:

50 kWh (battery capacity) x 0.95 (charging efficiency) = approximately 47.5 kWh

Since your solar system is expected to generate around 26.25 kWh tomorrow, it appears that your solar yield will 